# 从零复现 MobileNet 与 DenseNet：轻量卷积、倒残差和密集连接

这份 Notebook 不调用 `torchvision.models` 或 `timm`。我们只用基础张量算子和 `torch.nn` 的卷积、归一化、激活层，显式实现 `DepthwiseSeparableConv`、`MobileNetV1`、`InvertedResidual`、`MobileNetV2Tiny`、`DenseLayer`、`DenseBlock`、`Transition` 与 `DenseNetTiny.forward`。

重点不是把论文结构抄成一串层，而是回答工程复现时真正容易出错的问题：`groups` 到底隔离了哪些通道、什么时候能做残差相加、DenseNet 的通道数怎样增长、轻量化是否真的减少参数、BatchNorm 的 train/eval 状态是否污染线上结果，以及模型制品如何绑定结构和预处理。

所有数据离线合成、固定随机种子、CPU 单线程。微型任务只验证计算图能学习，不等价于 ImageNet 精度复现。

## 1. 两条设计路线与张量合同

MobileNet 用“先逐通道空间卷积、再逐点混合通道”降低计算量。对 $k\times k$ 卷积，普通卷积乘加量近似为

$$HWk^2C_{in}C_{out},$$

depthwise + pointwise 则约为

$$HW(k^2C_{in}+C_{in}C_{out}).$$

DenseNet 采取另一条路线：第 $\ell$ 层接收此前所有特征并只新增 $g$ 个通道，$x_\ell=H_\ell([x_0,\ldots,x_{\ell-1}])$。它改善特征复用，但拼接会增长激活内存。

本册统一输入为 `[N,C,H,W]`，分类 logits 为 `[N,num_classes]`。任何隐式广播、尺寸不匹配和错误通道数都应尽早失败。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from hashlib import sha256
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 340728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. Depthwise separable convolution：`groups` 不是装饰参数

`Conv2d(C,C,k,groups=C)` 把输入拆成 $C$ 个独立组：第 $c$ 个输出只能看到第 $c$ 个输入通道。它**没有通道混合能力**，所以随后必须用 $1\times1$ pointwise convolution 把 $C_{in}$ 映射到 $C_{out}$。

下面不仅检查 shape，还把 grouped convolution 与逐通道调用 `F.conv2d` 的结果逐元素对齐。这个 oracle 能抓住把 `groups` 写成 1、权重索引错位等实现错误。

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        if in_channels <= 0 or out_channels <= 0 or stride not in (1, 2):
            raise ValueError("invalid depthwise-separable configuration")
        self.in_channels = int(in_channels)
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride=stride,
                                   padding=1, groups=in_channels, bias=False)
        self.depth_bn = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.point_bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("expected NCHW with configured input channels")
        x = F.relu6(self.depth_bn(self.depthwise(x)))
        return F.relu6(self.point_bn(self.pointwise(x)))

# groups 数值 oracle：每个输出通道必须只依赖同编号输入通道。
oracle_x = torch.arange(1, 1 + 2 * 4 * 4, dtype=torch.float32).reshape(1, 2, 4, 4)
oracle_weight = torch.tensor([[[[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]]],
                              [[[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]]])
grouped = F.conv2d(oracle_x, oracle_weight, padding=1, groups=2)
separate = torch.cat([
    F.conv2d(oracle_x[:, c:c+1], oracle_weight[c:c+1], padding=1)
    for c in range(2)
], dim=1)
assert torch.equal(grouped, separate)

depth_probe = DepthwiseSeparableConv(3, 7, stride=2)
probe_x = torch.randn(2, 3, 16, 16, requires_grad=True)
probe_y = depth_probe(probe_x)
probe_y.square().mean().backward()
assert probe_y.shape == (2, 7, 8, 8)
assert depth_probe.depthwise.groups == 3
assert probe_x.grad is not None and torch.isfinite(probe_x.grad).all()
assert float(probe_x.grad.norm()) > 0

## 3. MobileNetV1：宽度、步幅和全局池化

MobileNetV1 的主体重复 depthwise separable block。每次 `stride=2` 同时改变空间分辨率；通道改变只发生在 pointwise 层。分类头使用 adaptive global average pooling，因此不会把某个固定 $H\times W$ 写死在 `Linear` 中。

小型版本保留论文核心算子，但减少 stage 和通道，便于 CPU 教学。`width_mult` 必须同时作用到相邻 block 的输入输出，否则下一层会收到错误通道数。

In [ ]:
class MobileNetV1(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, width_mult=0.5):
        super().__init__()
        if width_mult <= 0:
            raise ValueError("width_mult must be positive")
        channels = [max(4, int(c * width_mult)) for c in (16, 32, 64, 96)]
        self.in_channels = int(in_channels)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]), nn.ReLU6(inplace=False),
        )
        self.features = nn.Sequential(
            DepthwiseSeparableConv(channels[0], channels[1], stride=2),
            DepthwiseSeparableConv(channels[1], channels[1], stride=1),
            DepthwiseSeparableConv(channels[1], channels[2], stride=2),
            DepthwiseSeparableConv(channels[2], channels[3], stride=1),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(channels[3], num_classes)

    def forward_features(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("MobileNetV1 expected configured NCHW input")
        return self.features(self.stem(x))

    def forward(self, x):
        features = self.forward_features(x)
        return self.classifier(self.pool(features).flatten(1))

mobile_v1_probe = MobileNetV1()
mobile_v1_features = mobile_v1_probe.forward_features(torch.randn(4, 1, 16, 16))
mobile_v1_logits = mobile_v1_probe(torch.randn(4, 1, 16, 16))
assert mobile_v1_features.shape == (4, 48, 4, 4)
assert mobile_v1_logits.shape == (4, 3)
try:
    mobile_v1_probe(torch.randn(2, 3, 16, 16))
    raise AssertionError("channel mismatch must fail")
except ValueError:
    pass

## 4. MobileNetV2 倒残差：先扩张、depthwise、再线性压缩

倒残差块先用 $1\times1$ 把通道扩张 $t$ 倍，再做 depthwise spatial convolution，最后用**不带激活**的线性 pointwise 投影回输出通道。若最后也加 ReLU6，低维瓶颈中的信息更容易被截断。

只有 `stride == 1` 且 `in_channels == out_channels` 时，输入和分支 shape 完全一致，才能逐元素残差相加。下面把残差分支全部置零，直接验证输出必须等于输入；这比只检查 shape 更强。

In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expansion=2):
        super().__init__()
        if stride not in (1, 2) or expansion < 1:
            raise ValueError("invalid inverted residual configuration")
        hidden = int(in_channels * expansion)
        self.in_channels = int(in_channels)
        self.use_residual = stride == 1 and in_channels == out_channels
        layers = []
        if expansion != 1:
            layers += [nn.Conv2d(in_channels, hidden, 1, bias=False),
                       nn.BatchNorm2d(hidden), nn.ReLU6(inplace=False)]
        layers += [nn.Conv2d(hidden, hidden, 3, stride=stride, padding=1,
                             groups=hidden, bias=False),
                   nn.BatchNorm2d(hidden), nn.ReLU6(inplace=False),
                   nn.Conv2d(hidden, out_channels, 1, bias=False),
                   nn.BatchNorm2d(out_channels)]
        self.branch = nn.Sequential(*layers)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("inverted residual input mismatch")
        branch = self.branch(x)
        return x + branch if self.use_residual else branch

class MobileNetV2Tiny(nn.Module):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__()
        self.in_channels = in_channels
        self.stem = nn.Sequential(nn.Conv2d(in_channels, 12, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(12), nn.ReLU6())
        self.blocks = nn.Sequential(
            InvertedResidual(12, 12, 1, 1),
            InvertedResidual(12, 20, 2, 2),
            InvertedResidual(20, 20, 1, 2),
            InvertedResidual(20, 32, 2, 2),
        )
        self.head = nn.Linear(32, num_classes)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("MobileNetV2Tiny input mismatch")
        x = self.blocks(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(x, 1).flatten(1))

residual_oracle = InvertedResidual(6, 6, stride=1, expansion=2).eval()
with torch.no_grad():
    for parameter in residual_oracle.branch.parameters():
        parameter.zero_()
oracle_input = torch.randn(2, 6, 7, 7)
assert residual_oracle.use_residual
assert torch.equal(residual_oracle(oracle_input), oracle_input)
assert not InvertedResidual(6, 8, stride=1).use_residual
assert not InvertedResidual(6, 6, stride=2).use_residual
assert MobileNetV2Tiny()(torch.randn(3, 1, 16, 16)).shape == (3, 3)

## 5. DenseNet：拼接不是相加

若 block 输入通道为 $C_0$、有 $L$ 层、growth rate 为 $g$，输出通道严格为 $C_0+Lg$。每个 `DenseLayer` 只产生 $g$ 个新特征，然后通过 `torch.cat(..., dim=1)` 保留旧特征。若误写成加法，通道增长和特征复用都会消失。

Transition 使用 $1\times1$ 卷积压缩通道，再以平均池化降低分辨率。教学版使用 BN-ReLU-Conv 的 pre-activation 顺序。

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.in_channels = int(in_channels)
        self.growth_rate = int(growth_rate)
        self.norm = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, growth_rate, 3, padding=1, bias=False)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("DenseLayer input channel mismatch")
        new_features = self.conv(F.relu(self.norm(x)))
        return torch.cat([x, new_features], dim=1)

class DenseBlock(nn.Module):
    def __init__(self, in_channels, num_layers, growth_rate):
        super().__init__()
        if num_layers < 1 or growth_rate < 1:
            raise ValueError("invalid dense block")
        layers, channels = [], int(in_channels)
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate
        self.layers = nn.ModuleList(layers)
        self.out_channels = channels

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class Transition(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        if not 0 < out_channels <= in_channels:
            raise ValueError("transition must not expand channels")
        self.norm = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pool = nn.AvgPool2d(2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(F.relu(self.norm(x))))

class DenseNetTiny(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, growth_rate=6):
        super().__init__()
        self.in_channels = in_channels
        self.stem = nn.Conv2d(in_channels, 12, 3, padding=1, bias=False)
        self.block1 = DenseBlock(12, 3, growth_rate)
        compressed = self.block1.out_channels // 2
        self.transition = Transition(self.block1.out_channels, compressed)
        self.block2 = DenseBlock(compressed, 3, growth_rate)
        self.norm = nn.BatchNorm2d(self.block2.out_channels)
        self.classifier = nn.Linear(self.block2.out_channels, num_classes)

    def forward_features(self, x):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("DenseNetTiny input mismatch")
        x = self.transition(self.block1(self.stem(x)))
        return self.block2(x)

    def forward(self, x):
        x = F.relu(self.norm(self.forward_features(x)))
        return self.classifier(F.adaptive_avg_pool2d(x, 1).flatten(1))

dense_probe = DenseBlock(5, num_layers=4, growth_rate=3)
dense_output = dense_probe(torch.randn(2, 5, 8, 8, requires_grad=True))
assert dense_probe.out_channels == 17
assert dense_output.shape == (2, 17, 8, 8)
dense_output.mean().backward()
assert all(layer.conv.weight.grad is not None for layer in dense_probe.layers)
dense_net_probe = DenseNetTiny()
assert dense_net_probe.forward_features(torch.randn(2, 1, 16, 16)).shape == (2, 33, 8, 8)
assert dense_net_probe(torch.randn(2, 1, 16, 16)).shape == (2, 3)

## 6. 参数量与理论计算量：先定义比较口径

把一个 $3\times3$ 普通卷积与同尺寸的 depthwise + pointwise 比较，参数量分别为 $9C_{in}C_{out}$ 与 $9C_{in}+C_{in}C_{out}$（此处均无 bias）。比例不包含 BN、激活和内存访问，因此参数少不保证在所有硬件上延迟更低。

端到端模型比较也必须用相同输入通道和类别数；不同宽度、分辨率或分类头会改变结论。

In [ ]:
def parameter_count(module):
    return sum(parameter.numel() for parameter in module.parameters())

cin, cout, kernel = 16, 32, 3
standard = nn.Conv2d(cin, cout, kernel, padding=1, bias=False)
separable_core = nn.Sequential(
    nn.Conv2d(cin, cin, kernel, padding=1, groups=cin, bias=False),
    nn.Conv2d(cin, cout, 1, bias=False),
)
expected_standard = kernel * kernel * cin * cout
expected_separable = kernel * kernel * cin + cin * cout
assert parameter_count(standard) == expected_standard
assert parameter_count(separable_core) == expected_separable
assert expected_separable < expected_standard / 5

model_counts = {
    "MobileNetV1": parameter_count(MobileNetV1()),
    "MobileNetV2Tiny": parameter_count(MobileNetV2Tiny()),
    "DenseNetTiny": parameter_count(DenseNetTiny()),
}
assert all(count > 0 for count in model_counts.values())
print({"conv_parameter_ratio": expected_separable / expected_standard,
       "tiny_model_parameters": model_counts})

## 7. 受控学习任务与无泄漏切分

我们合成三类 $16\times16$ 单通道图像：竖条、横条和十字。每张图的位置、宽度和噪声独立变化。先按固定索引生成 train/validation/test，再只用 train 的均值方差做标准化，避免把 validation/test 统计量泄漏进训练。

这是“结构能否反向传播并学会空间模式”的 smoke test。重复纹理比自然图像简单得多，不能据此比较 MobileNet 和 DenseNet 的真实泛化能力。

In [ ]:
def make_pattern_dataset(count, seed):
    generator = torch.Generator().manual_seed(seed)
    images = torch.zeros(count, 1, 16, 16)
    labels = torch.arange(count) % 3
    for index, label in enumerate(labels.tolist()):
        position = int(torch.randint(4, 12, (1,), generator=generator))
        width = int(torch.randint(1, 3, (1,), generator=generator))
        if label in (0, 2):
            images[index, 0, :, position:position + width] = 1.0
        if label in (1, 2):
            images[index, 0, position:position + width, :] = 1.0
    images += 0.08 * torch.randn(images.shape, generator=generator)
    return images, labels.long()

train_raw, train_y = make_pattern_dataset(72, SEED + 1)
valid_raw, valid_y = make_pattern_dataset(30, SEED + 2)
test_raw, test_y = make_pattern_dataset(30, SEED + 3)
train_mean, train_std = train_raw.mean(), train_raw.std().clamp_min(1e-6)
normalize = lambda tensor: (tensor - train_mean) / train_std
train_x, valid_x, test_x = map(normalize, (train_raw, valid_raw, test_raw))

assert train_x.shape == (72, 1, 16, 16)
assert set(train_y.tolist()) == set(valid_y.tolist()) == set(test_y.tolist()) == {0, 1, 2}
assert abs(float(train_x.mean())) < 1e-6
assert not torch.isclose(valid_raw.mean(), train_mean, atol=0, rtol=0)

## 8. 训练、梯度与 validation checkpoint

为了控制运行时间，训练小型 MobileNetV2。每一步使用完整 train set；validation 只负责选择 checkpoint，test 只在选择完成后读取一次。第一步显式检查所有可训练路径产生有限梯度，训练结束比较受控任务的初末损失和独立 test accuracy。

真实训练应使用 mini-batch、数据增强、学习率调度、混合精度和多次种子，并报告均值与方差；这里故意不把 test 用于调参。

In [ ]:
torch.manual_seed(SEED)
model34 = MobileNetV2Tiny().to(DEVICE)
optimizer = torch.optim.Adam(model34.parameters(), lr=0.02)
history, best_validation, best_state = [], float("inf"), None

for step in range(81):
    model34.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model34(train_x)
    loss = F.cross_entropy(logits, train_y)
    loss.backward()
    if step == 0:
        first_grad_norm = torch.sqrt(sum(
            parameter.grad.detach().square().sum()
            for parameter in model34.parameters() if parameter.grad is not None
        ))
    optimizer.step()
    history.append(float(loss.detach()))
    if step % 5 == 0:
        model34.eval()
        with torch.no_grad():
            validation_loss = float(F.cross_entropy(model34(valid_x), valid_y))
        if validation_loss < best_validation:
            best_validation = validation_loss
            best_state = deepcopy(model34.state_dict())

assert best_state is not None
model34.load_state_dict(best_state)
model34.eval()
with torch.no_grad():
    test_logits = model34(test_x)
test_accuracy = float((test_logits.argmax(1) == test_y).float().mean())
assert torch.isfinite(first_grad_norm) and float(first_grad_norm) > 0
assert min(history[-10:]) < history[0] * 0.35
assert test_accuracy >= 0.90
print({"train_loss_first_last": [history[0], history[-1]],
       "best_validation_loss": best_validation, "test_accuracy": test_accuracy})

## 9. BatchNorm 状态合同：推理必须 `eval()`

BatchNorm 在 train 模式用当前 batch 统计量并更新 `running_mean/running_var`；在 eval 模式使用冻结的 running statistics。线上漏掉 `eval()` 会让同一个样本随请求 batch 组成变化，还会悄悄改变模型状态。

下面在模型副本上分别执行 eval 和 train 前向：eval 不得修改 running mean 且重复结果一致，train 则应更新状态。使用副本避免这项探针污染已选择的 checkpoint。

In [ ]:
bn_probe_model = deepcopy(model34)
first_bn = next(module for module in bn_probe_model.modules() if isinstance(module, nn.BatchNorm2d))
before_eval = first_bn.running_mean.clone()
bn_probe_model.eval()
with torch.no_grad():
    eval_a = bn_probe_model(test_x[:4])
    eval_b = bn_probe_model(test_x[:4])
after_eval = first_bn.running_mean.clone()
assert torch.equal(before_eval, after_eval)
assert torch.equal(eval_a, eval_b)

bn_probe_model.train()
with torch.no_grad():
    _ = bn_probe_model(torch.full_like(train_x[:8], 4.0))
after_train = first_bn.running_mean.clone()
assert not torch.equal(after_eval, after_train)

# 训练模式下，单样本输出依赖同 batch 的其他样本；eval 不应如此。
bn_probe_model.eval()
with torch.no_grad():
    alone = bn_probe_model(test_x[:1])
    in_batch = bn_probe_model(torch.cat([test_x[:1], test_x[1:8]], dim=0))[:1]
assert torch.allclose(alone, in_batch, atol=1e-6)

## 10. 制品合同：发布者信任锚、训练快照与可重建预处理

制品自己保存 `manifest_sha256` 只能发现意外损坏：攻击者若整体替换权重、manifest 和内部摘要，它们仍然“自洽”。本节把发布者侧只读登记表 `artifact_id/version -> expected bundle digest` 当作 package 外的信任锚。bundle digest 同时覆盖 canonical manifest 与 state 中每个 tensor 的 `key/dtype/shape/bytes`；内部 hash 全部重签也不能改变发布登记值。

语义校验同样不能停在“能 strict load”：类别必须唯一非空且 `num_classes == len(class_names)`，config、输入 shape、state schema 和标准化 recipe 都要精确匹配。train/validation/test 的原始图像、target、count 与 seed 被绑定为 split snapshot；因为这里的数据生成受控，loader 会重新生成 train split，并只由这个绑定快照重算 mean/std。生产中信任锚应来自签名发布元数据、透明日志或只读制品服务，而不是和模型一起放在可替换目录。

In [ ]:
def canonical_json34(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False).encode("utf-8")

def _feed_digest34(hasher, payload):
    hasher.update(len(payload).to_bytes(8, "big"))
    hasher.update(payload)

def clone_state34(state_dict):
    if not hasattr(state_dict, "items"):
        raise ValueError("state_dict must be a mapping")
    cloned = {}
    for key, tensor in state_dict.items():
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("state entries must be string -> Tensor")
        cloned[key] = tensor.detach().cpu().contiguous().clone()
    return cloned

def _update_state_digest34(hasher, state_dict):
    if not isinstance(state_dict, dict) or not state_dict:
        raise ValueError("artifact state_dict must be a non-empty plain dict")
    for key in sorted(state_dict):
        tensor = state_dict[key]
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("invalid state entry")
        cpu = tensor.detach().cpu().contiguous()
        header = canonical_json34({"key": key, "dtype": str(cpu.dtype),
                                   "shape": list(cpu.shape)})
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()
        _feed_digest34(hasher, header)
        _feed_digest34(hasher, raw)

def canonical_state_digest34(state_dict):
    hasher = sha256()
    _feed_digest34(hasher, b"canonical-state-dict-v1")
    _update_state_digest34(hasher, state_dict)
    return hasher.hexdigest()

def canonical_bundle_digest34(manifest, state_dict):
    hasher = sha256()
    _feed_digest34(hasher, b"canonical-model-bundle-v1")
    _feed_digest34(hasher, canonical_json34(manifest))
    _update_state_digest34(hasher, state_dict)
    return hasher.hexdigest()

def state_schema34(state_dict):
    return [{"key": key, "dtype": str(state_dict[key].dtype),
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]

def expected_pattern_release34():
    split_specs = {"train": (72, SEED + 1), "validation": (30, SEED + 2),
                   "test": (30, SEED + 3)}
    splits, rebuilt = {}, {}
    for name, (count, seed) in split_specs.items():
        images, labels = make_pattern_dataset(count, seed)
        rebuilt[name] = (images, labels)
        snapshot_state = clone_state34({"images": images, "labels": labels})
        splits[name] = {"count": count, "seed": seed,
                        "images_labels_sha256": canonical_state_digest34(snapshot_state)}
    train_images = rebuilt["train"][0]
    std_floor = 1e-6
    derived_mean = float(train_images.mean())
    derived_std = float(train_images.std().clamp_min(std_floor))
    data_contract = {"dataset_recipe": "controlled-bars-v1", "splits": splits,
                     "target_schema": "int64 class index"}
    normalizer = {"kind": "standard-score", "fit_split": "train",
                  "mean": derived_mean, "std": derived_std, "std_floor": std_floor}
    return data_contract, normalizer

EXPECTED_CONFIG34 = {"in_channels": 1, "num_classes": 3}
EXPECTED_CLASSES34 = ["vertical", "horizontal", "cross"]
EXPECTED_INPUT_SHAPE34 = [1, 16, 16]
EXPECTED_PREPROCESS34 = {"recipe": "standardize-from-bound-train-v1",
                         "formula": "(x-mean)/max(std,std_floor)",
                         "layout": "NCHW", "dtype": "float32"}
ARTIFACT_ID34, ARTIFACT_VERSION34 = "vision.controlled-mobilenetv2", "1.0.0"

def build_artifact(model, state_dict):
    if type(model) is not MobileNetV2Tiny:
        raise ValueError("publisher only accepts the audited MobileNetV2Tiny class")
    config = {"in_channels": int(model.in_channels),
              "num_classes": int(model.head.out_features)}
    state = clone_state34(state_dict)
    probe = MobileNetV2Tiny(**config)
    probe.load_state_dict(state, strict=True)
    data_contract, normalizer = expected_pattern_release34()
    manifest = {
        "schema_version": 2, "artifact_id": ARTIFACT_ID34,
        "artifact_version": ARTIFACT_VERSION34, "architecture": "MobileNetV2Tiny",
        "architecture_config": config, "model_state_schema": state_schema34(state),
        "class_names": list(EXPECTED_CLASSES34), "input_shape": list(EXPECTED_INPUT_SHAPE34),
        "preprocess_recipe": deepcopy(EXPECTED_PREPROCESS34), "normalizer": normalizer,
        "data_contract": data_contract, "torch_version": torch.__version__,
        "state_digest_sha256": canonical_state_digest34(state),
    }
    return {"manifest": manifest,
            "manifest_sha256": sha256(canonical_json34(manifest)).hexdigest(),
            "bundle_sha256": canonical_bundle_digest34(manifest, state),
            "state_dict": state}

def validate_mobile_contract34(manifest, state_dict):
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",
                "architecture_config", "model_state_schema", "class_names", "input_shape",
                "preprocess_recipe", "normalizer", "data_contract", "torch_version",
                "state_digest_sha256"}
    if set(manifest) != required or manifest["schema_version"] != 2:
        raise ValueError("manifest schema mismatch")
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID34, ARTIFACT_VERSION34):
        raise ValueError("artifact identity mismatch")
    if (manifest["architecture"] != "MobileNetV2Tiny" or
            manifest["architecture_config"] != EXPECTED_CONFIG34):
        raise ValueError("architecture/config mismatch")
    names = manifest["class_names"]
    if (names != EXPECTED_CLASSES34 or len(names) != manifest["architecture_config"]["num_classes"] or
            len(set(names)) != len(names) or any(not isinstance(x, str) or not x.strip() for x in names)):
        raise ValueError("class mapping must be exact, unique and non-empty")
    if manifest["input_shape"] != EXPECTED_INPUT_SHAPE34:
        raise ValueError("input shape mismatch")
    if manifest["preprocess_recipe"] != EXPECTED_PREPROCESS34:
        raise ValueError("preprocess recipe mismatch")
    expected_data, expected_normalizer = expected_pattern_release34()
    if manifest["data_contract"] != expected_data:
        raise ValueError("train/target/split snapshot mismatch")
    normalizer = manifest["normalizer"]
    if (normalizer != expected_normalizer or not math.isfinite(normalizer["mean"]) or
            not math.isfinite(normalizer["std"]) or normalizer["std"] <= 0):
        raise ValueError("normalizer is not derived from the bound train snapshot")
    expected_schema = state_schema34(MobileNetV2Tiny(**EXPECTED_CONFIG34).state_dict())
    if manifest["model_state_schema"] != expected_schema or state_schema34(state_dict) != expected_schema:
        raise ValueError("model state schema mismatch")

def load_trusted_mobile(artifact):
    if not isinstance(artifact, dict) or set(artifact) != {
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:
        raise ValueError("artifact package schema mismatch")
    manifest, state = artifact["manifest"], artifact["state_dict"]
    if not isinstance(manifest, dict):
        raise ValueError("manifest must be a dict")
    actual_bundle = canonical_bundle_digest34(manifest, state)
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))
    expected_bundle = PUBLISHER_REGISTRY34.get(identity)
    if expected_bundle is None or actual_bundle != expected_bundle:
        raise ValueError("publisher registry rejected this bundle")
    if artifact["bundle_sha256"] != actual_bundle:
        raise ValueError("internal bundle digest mismatch")
    if sha256(canonical_json34(manifest)).hexdigest() != artifact["manifest_sha256"]:
        raise ValueError("manifest digest mismatch")
    if canonical_state_digest34(state) != manifest["state_digest_sha256"]:
        raise ValueError("canonical state digest mismatch")
    validate_mobile_contract34(manifest, state)
    loaded = MobileNetV2Tiny(**manifest["architecture_config"])
    loaded.load_state_dict(state, strict=True)
    return loaded.eval()

def resign_inside34(artifact):
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest34(artifact["state_dict"])
    artifact["manifest_sha256"] = sha256(canonical_json34(artifact["manifest"])).hexdigest()
    artifact["bundle_sha256"] = canonical_bundle_digest34(artifact["manifest"], artifact["state_dict"])
    return artifact

artifact34 = build_artifact(model34, model34.state_dict())
PUBLISHER_REGISTRY34 = MappingProxyType({
    (ARTIFACT_ID34, ARTIFACT_VERSION34): artifact34["bundle_sha256"]
})
loaded34 = load_trusted_mobile(artifact34)
with torch.no_grad():
    assert torch.equal(loaded34(test_x[:5]), model34(test_x[:5]))

# 整体替换为四分类模型，并重签 package 内全部摘要；外部发布登记仍拒绝。
forged_model34 = MobileNetV2Tiny(num_classes=4)
forged_four_class34 = deepcopy(artifact34)
forged_four_class34["state_dict"] = clone_state34(forged_model34.state_dict())
forged_four_class34["manifest"]["architecture_config"]["num_classes"] = 4
forged_four_class34["manifest"]["class_names"] = ["vertical", "horizontal", "cross", "other"]
forged_four_class34["manifest"]["model_state_schema"] = state_schema34(forged_four_class34["state_dict"])
resign_inside34(forged_four_class34)
assert forged_four_class34["bundle_sha256"] == canonical_bundle_digest34(
    forged_four_class34["manifest"], forged_four_class34["state_dict"])
try:
    load_trusted_mobile(forged_four_class34)
    raise AssertionError("self-signed four-class replacement must fail")
except ValueError:
    pass

# 权重不变但把 train-only mean 平移 7，同样重签内部摘要，仍不能越过发布信任锚。
forged_mean34 = deepcopy(artifact34)
forged_mean34["manifest"]["normalizer"]["mean"] += 7.0
resign_inside34(forged_mean34)
try:
    load_trusted_mobile(forged_mean34)
    raise AssertionError("self-signed normalizer replacement must fail")
except ValueError:
    pass

try:
    PUBLISHER_REGISTRY34[(ARTIFACT_ID34, ARTIFACT_VERSION34)] = forged_mean34["bundle_sha256"]
    raise AssertionError("publisher registry must be immutable")
except TypeError:
    pass

## 11. 常见失败模式、生产差距与复现清单

1. **depthwise 后漏掉 pointwise**：空间特征更新了，但不同通道永远不能交流。
2. **倒残差无条件相加**：stride 或通道改变时不是残差连接，应直接拒绝或使用显式 projection；MobileNetV2 原块选择前者。
3. **DenseNet 写成相加**：growth-rate 合同被破坏；应逐层断言通道为 $C_0+\ell g$。
4. **参数少等同于延迟低**：depthwise 算子可能受内存访问、kernel 实现和设备支持限制；必须在目标硬件做 warm-up 后测 p50/p95/p99。
5. **BN 探针污染模型**：所有校准/诊断在副本上做；小 batch 可评估 GroupNorm、SyncBN 或冻结 BN。
6. **只保存权重或自签 hash**：必须用发布者外部信任锚绑定权重、数据快照、预处理、标签和结构。

生产复现还需真实数据治理、分组切分、增强策略、蒸馏/量化校准、AMP 数值审计、多随机种子、吞吐与峰值内存、OOD/鲁棒性/公平性测试。导出 ONNX 或移动端格式后必须逐层或端到端对齐数值，不能只看导出成功。

### 论文来源

- Howard et al., [*MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications*](https://arxiv.org/abs/1704.04861), 2017.
- Sandler et al., [*MobileNetV2: Inverted Residuals and Linear Bottlenecks*](https://arxiv.org/abs/1801.04381), 2018.
- Huang et al., [*Densely Connected Convolutional Networks*](https://arxiv.org/abs/1608.06993), CVPR 2017.

本册复现核心计算图和工程合同；没有声称复现论文训练配方、数据规模或榜单结果。